# Topic 5: Model Evaluation and SHAP Explainability
**Module 1 - Introduction to Machine Learning in Python**


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, roc_curve, average_precision_score,
                             precision_recall_curve)
import matplotlib.pyplot as plt


## 1. Prepare Data and Train Models


In [ ]:
np.random.seed(42)
n = 10000
X = pd.DataFrame({
    'ext_source_1': np.random.beta(3, 3, n),
    'ext_source_2': np.random.beta(4, 2, n),
    'credit_income_ratio': np.random.lognormal(1, 0.5, n),
    'age': np.random.normal(45, 12, n).clip(21, 80),
    'dti': np.random.uniform(0, 50, n),
})
logit = -2 - 3*X['ext_source_1'] - 2*X['ext_source_2'] + 0.5*X['credit_income_ratio'] - 0.02*X['age'] + 0.03*X['dti'] + np.random.normal(0, 1, n)
from scipy.special import expit
y = (np.random.random(n) < expit(logit)).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

rf = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)
rf_proba = rf.predict_proba(X_test)[:, 1]

print(f'AUROC: {roc_auc_score(y_test, rf_proba):.4f}')


## 2. Complete Financial Metrics Table


In [ ]:
auroc = roc_auc_score(y_test, rf_proba)
gini = 2 * auroc - 1
fpr, tpr, _ = roc_curve(y_test, rf_proba)
ks = max(tpr - fpr)
auprc = average_precision_score(y_test, rf_proba)

metrics = pd.DataFrame([{
    'Metric': 'AUROC', 'Value': f'{auroc:.4f}', 'Interpretation': f'Model discriminates well' if auroc > 0.7 else 'Weak',
}, {
    'Metric': 'Gini', 'Value': f'{gini:.4f}', 'Interpretation': f'Good' if gini > 0.3 else 'Weak',
}, {
    'Metric': 'KS', 'Value': f'{ks:.4f}', 'Interpretation': f'Good separation' if ks > 0.3 else 'Moderate',
}, {
    'Metric': 'AUPRC', 'Value': f'{auprc:.4f}', 'Interpretation': f'Above baseline ({y_test.mean():.4f})',
}])
print('Financial Metrics:')
print(metrics.to_string(index=False))


## 3. Cross-Validation Stability


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf, X_train, y_train, cv=cv, scoring='roc_auc')

print('CV AUROC per fold:')
for i, s in enumerate(cv_scores, 1):
    print(f'  Fold {i}: {s:.4f}')
print(f'  Mean:   {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')
print(f'  Stable: {"Yes" if cv_scores.std() < 0.02 else "Investigate variance"}')


## 4. SHAP Analysis


In [ ]:
import shap

# Compute SHAP values
explainer = shap.TreeExplainer(rf)
X_sample = X_test.iloc[:500]  # Sample for speed
shap_values = explainer.shap_values(X_sample)

# If binary, shap_values is a list [class_0, class_1]
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

# Summary plot
shap.summary_plot(sv, X_sample, show=False)
plt.tight_layout()
plt.show()


## 5. Individual SHAP Waterfall Plots


In [ ]:
y_proba_sample = rf.predict_proba(X_sample)[:, 1]
ev = explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value

high_risk = np.argmax(y_proba_sample)
low_risk = np.argmin(y_proba_sample)
borderline = np.argmin(np.abs(y_proba_sample - 0.3))

for name, idx in [('High Risk', high_risk), ('Borderline', borderline), ('Low Risk', low_risk)]:
    print(f'\n{name} Applicant - PD: {y_proba_sample[idx]:.4f}')
    explanation = shap.Explanation(
        values=sv[idx],
        base_values=ev,
        data=X_sample.iloc[idx],
        feature_names=X_sample.columns.tolist()
    )
    shap.waterfall_plot(explanation, max_display=10, show=False)
    plt.title(f'{name} (PD={y_proba_sample[idx]:.4f})')
    plt.tight_layout()
    plt.show()


## 6. Population Stability Index (PSI)


In [ ]:
def compute_psi(expected, actual, bins=10):
    '''Compute PSI between two distributions'''
    breakpoints = np.quantile(expected.dropna(), np.linspace(0, 1, bins + 1))
    breakpoints[0] = -np.inf
    breakpoints[-1] = np.inf
    exp_counts = np.histogram(expected.dropna(), breakpoints)[0]
    act_counts = np.histogram(actual.dropna(), breakpoints)[0]
    exp_pct = np.clip(exp_counts / exp_counts.sum(), 0.001, None)
    act_pct = np.clip(act_counts / act_counts.sum(), 0.001, None)
    return np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct))

psi_results = []
for col in X.columns:
    psi = compute_psi(X_train[col], X_test[col])
    status = 'Stable' if psi < 0.1 else ('Moderate Shift' if psi < 0.25 else 'REVIEW NEEDED')
    psi_results.append({'Feature': col, 'PSI': round(psi, 4), 'Status': status})

print('Population Stability Index:')
print(pd.DataFrame(psi_results).to_string(index=False))
